# 06: Métricas de risco

Seis métricas para avaliar o risco de uma ação da B3, todas calculadas com a WEGE3 como exemplo.

| Métrica | Pergunta que responde |
|---|---|
| Sharpe Ratio | Quanto retorno eu ganho por unidade de risco? |
| Sortino Ratio | A mesma coisa, mas penalizando só as quedas |
| Maximum Drawdown | Qual foi a pior queda histórica? |
| VaR Monte Carlo | Qual a minha perda máxima com 95% de confiança? |
| Beta / Alpha | A ação é agressiva ou defensiva? Bate o benchmark? |
| Correlação global | Se o mercado global cai, essa ação também cai? |

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

In [2]:
import yfinance as yf
import numpy as np
import pandas as pd

TICKER = 'WEGE3.SA'
BENCHMARK = '^BVSP'
SELIC_ANUAL = 0.1075  # Selic atual (10.75% a.a.)
PERIODO = '1y'

acao = yf.Ticker(TICKER)
ibov = yf.Ticker(BENCHMARK)

hist = acao.history(period=PERIODO)
hist_ibov = ibov.history(period=PERIODO)

retornos = hist['Close'].pct_change().dropna()
retornos_ibov = hist_ibov['Close'].pct_change().dropna()

# Alinha os indices (mesmos dias de negociacao)
df = pd.DataFrame({'wege3': retornos, 'ibov': retornos_ibov}).dropna()

print(f'Empresa: {acao.info.get("longName", TICKER)}')
print(f'Periodo: {df.index[0].date()} ate {df.index[-1].date()}')
print(f'Dias uteis: {len(df)}')
print(f'Retorno acumulado WEGE3: {(1 + df["wege3"]).prod() - 1:.1%}')
print(f'Retorno acumulado IBOV:  {(1 + df["ibov"]).prod() - 1:.1%}')

Empresa: WEG S.A.
Periodo: 2025-11-07 ate 2026-11-06
Dias uteis: 247
Retorno acumulado WEGE3: 14.4%
Retorno acumulado IBOV:  9.9%


## 1. Sharpe Ratio

**O que é?** Retorno ajustado ao risco. Responde: *"Quanto de retorno eu ganho por unidade de risco que tomo?"*

**Fórmula:**
```
Sharpe = (Retorno médio - Taxa sem risco) / Volatilidade
```

**Referência:**
```
Sharpe > 1.0    -> Bom (você ganha mais do que o risco)
Sharpe 0.5-1.0  -> OK
Sharpe < 0.5    -> Fraco (risco alto para pouco retorno)
```

**Por que usar a Selic como taxa sem risco?**
A Selic é o retorno garantido no Brasil (renda fixa). Se uma ação não bate a Selic depois de ajustar pelo risco, não vale o risco de tê-la na carteira.

In [3]:
# Converte Selic anual para diaria (252 dias uteis)
selic_diaria = (1 + SELIC_ANUAL) ** (1/252) - 1

retorno_medio = df['wege3'].mean()
volatilidade = df['wege3'].std()

sharpe_diario = (retorno_medio - selic_diaria) / volatilidade
sharpe_anualizado = sharpe_diario * np.sqrt(252)

def interpretar_sharpe(valor: float) -> str:
    """Retorna 'Bom', 'OK' ou 'Fraco' baseado nos benchmarks acima."""
    if valor > 1.0:
        return 'Bom'
    elif valor >= 0.5:
        return 'OK'
    else:
        return 'Fraco'

print(f'Retorno medio diario: {retorno_medio:.4%}')
print(f'Volatilidade diaria:  {volatilidade:.4%}')
print(f'Selic diaria:         {selic_diaria:.6%}')
print(f'Sharpe anualizado:    {sharpe_anualizado:.4f}')
print(f'Interpretacao:        {interpretar_sharpe(sharpe_anualizado)}')

Retorno medio diario: 0.0734%
Volatilidade diaria:  1.9430%
Selic diaria:         0.040526%
Sharpe anualizado:    0.2684
Interpretacao:        Fraco


## 2. Sortino Ratio

**O que é?** Igual ao Sharpe, mas **só penaliza a volatilidade negativa** (as quedas).

**Por que existe?**
O Sharpe penaliza subidas e descidas do mesmo jeito. Uma ação que sobe muito parece volátil no Sharpe, mas no Sortino ela é tratada melhor, porque essa volatilidade vem de alta.

**Fórmula:**
```
Sortino = (Retorno médio - Taxa sem risco) / Volatilidade negativa
```

**Regra prática:**
```
Sortino > Sharpe  -> A ação sobe mais do que cai (assimetria positiva)
Sortino = Sharpe  -> Sobe e cai na mesma medida
```

In [4]:
retornos_wege3 = df['wege3']

# Apenas os dias de queda
retornos_negativos = retornos_wege3[retornos_wege3 < 0]
downside_std = retornos_negativos.std()

sortino_diario = (retorno_medio - selic_diaria) / downside_std
sortino_anualizado = sortino_diario * np.sqrt(252)

print(f'Volatilidade total:   {volatilidade:.4%}  (usada no Sharpe)')
print(f'Volatilidade quedas:  {downside_std:.4%}  (usada no Sortino)')
print()
print(f'Sharpe anualizado:    {sharpe_anualizado:.4f}')
print(f'Sortino anualizado:   {sortino_anualizado:.4f}')
print(f'Sortino > Sharpe:     {sortino_anualizado > sharpe_anualizado}')
print(f'-> Assimetria positiva: {sortino_anualizado > sharpe_anualizado}')

Volatilidade total:   1.9430%  (usada no Sharpe)
Volatilidade quedas:  1.2434%  (usada no Sortino)

Sharpe anualizado:    0.2684
Sortino anualizado:   0.4194
Sortino > Sharpe:     True
-> Assimetria positiva: True


## 3. Maximum Drawdown (MDD)

**O que é?** A maior queda histórica do **pico ao vale**.

**Fórmula:**
```
MDD = (Pico - Vale) / Pico
```

**Exemplo:**
```
Preço: 50 -> 55 (pico) -> 40 (vale) -> 60

MDD = (55 - 40) / 55 = 27.3%
```

**Na prática:** responde *"qual a pior queda que eu teria sofrido se comprasse no pior momento?"*
Se você não aguenta ver -15% na carteira, uma ação com MDD de -30% não é para você.

**Referência:**
```
MDD > -10%        -> Baixo risco
MDD -10% a -20%   -> Risco moderado
MDD < -20%        -> Risco alto
```

In [5]:
precos = hist['Close']

# Pico acumulado ate cada momento (maior preco visto ate aqui)
pico_acumulado = precos.cummax()

# Drawdown: queda percentual em relacao ao pico mais recente
drawdown_serie = (precos - pico_acumulado) / pico_acumulado

# O menor valor (mais negativo) e o Maximum Drawdown
mdd = drawdown_serie.min()
data_mdd = drawdown_serie.idxmin()

print(f'Maximum Drawdown:  {mdd:.1%}')
print(f'Data do pior vale: {data_mdd.date()}')
print()
print(f'Drawdown atual (em relacao ao ultimo pico):')
print(f'  {drawdown_serie.iloc[-1]:.1%}')

Maximum Drawdown:  -22.5%
Data do pior vale: 2026-06-03

Drawdown atual (em relacao ao ultimo pico):
  -10.2%


## 4. VaR Monte Carlo

**O que é?** Uma simulação: *"Em 95% dos cenários, qual a minha perda máxima esperada em 1 dia?"*

**Por que Monte Carlo?**
Em vez de olhar só o histórico, que pode ser curto, simulamos 10.000 cenários possíveis a partir da distribuição dos retornos históricos (média e desvio padrão como parâmetros).

**Como interpretar:**
```
Você tem R$100.000 em WEGE3.
VaR 95% = -3.5%

-> Em 95 de 100 dias, você perde menos de R$3.500.
-> Em 5 de 100 dias, você perde mais que R$3.500.
```

**Referência:**
```
VaR -2% a -4%  -> Normal para ações
VaR < -5%      -> Ação muito volátil
```

In [6]:
# Parametros da distribuicao historica
media_ret = df['wege3'].mean()
std_ret = df['wege3'].std()

# Simula 10.000 retornos possiveis (distribuicao normal com parametros historicos)
np.random.seed(42)
simulacoes = np.random.normal(media_ret, std_ret, 10_000)

# Percentil 5 captura os piores 5% dos cenarios (confianca de 95%)
var_95 = np.percentile(simulacoes, 5)

investimento = 100_000
perda_maxima = investimento * abs(var_95)

print(f'VaR 95% (diario):      {var_95:.2%}')
print(f'Em R$100.000 investido:')
print(f'  Perda maxima diaria: R${perda_maxima:,.0f}')
print(f'  (com 95% de confianca)')

VaR 95% (diario):      -3.14%
Em R$100.000 investido:
  Perda maxima diaria: R$3,142
  (com 95% de confianca)


## 5. Beta e Alpha

### Beta
**O que é?** Sensibilidade ao mercado: *"Se o IBOV sobe 10%, quanto a WEGE3 sobe?"*

```
Beta = Covariância(WEGE3, IBOV) / Variância(IBOV)

Beta = 1.0  -> Anda junto com o IBOV
Beta = 1.5  -> Sobe 15% quando o IBOV sobe 10% (agressiva)
Beta = 0.8  -> Sobe 8% quando o IBOV sobe 10% (defensiva)
Beta < 0.0  -> Anda no sentido contrário (raro)
```

### Alpha
**O que é?** O retorno **acima do esperado** para o beta da ação (modelo CAPM).

```
Retorno esperado = Selic + Beta x (IBOV - Selic)
Alpha = Retorno real - Retorno esperado

Alpha > 0  -> A ação rendeu mais do que o risco justificava
Alpha < 0  -> A ação decepcionou para o risco que tem
```

In [7]:
# Beta via matriz de covariancia
cov_matrix = df.cov()
beta = cov_matrix.loc['wege3', 'ibov'] / df['ibov'].var()

# Alpha via CAPM
retorno_anual_wege3 = (1 + df['wege3'].mean()) ** 252 - 1
retorno_anual_ibov = (1 + df['ibov'].mean()) ** 252 - 1
retorno_esperado = SELIC_ANUAL + beta * (retorno_anual_ibov - SELIC_ANUAL)
alpha = retorno_anual_wege3 - retorno_esperado

print(f'Beta calculado:              {beta:.4f}')
print(f'Beta yfinance (referencia):  {acao.info.get("beta", "N/A")}')
print()
print(f'Retorno real WEGE3 (anual):  {retorno_anual_wege3:.1%}')
print(f'Retorno esperado (CAPM):     {retorno_esperado:.1%}')
print(f'Alpha anual:                 {alpha:.1%}')
print(f'Acima do esperado:           {alpha > 0}')

Beta calculado:              0.2924
Beta yfinance (referencia):  -0.033

Retorno real WEGE3 (anual):  20.3%
Retorno esperado (CAPM):     11.6%
Alpha anual:                 8.8%
Acima do esperado:           True


## 6. Correlação com índices globais

**O que é?** O quanto a WEGE3 se move junto com os mercados internacionais.

```
Correlação vai de -1.0 a +1.0

 1.0 -> Movem-se igual
 0.5 -> Correlação moderada
 0.0 -> Sem relação
-0.5 -> Movem-se em sentidos opostos
```

**Para que serve?**
Diversificação. Se a WEGE3 e o S&P 500 têm correlação de 0.8, ter as duas não protege muito a carteira. Uma ação com correlação baixa com o mercado global diversifica melhor.

**Referência:**
```
Correlação > 0.7   -> Alta dependência do mercado global
Correlação 0.3-0.7 -> Dependência moderada
Correlação < 0.3   -> Mais independente (bom para diversificar)
```

In [8]:
indices_globais = {
    'S&P 500': '^GSPC',
    'Nikkei':  '^N225',
    'FTSE':    '^FTSE',
    'DAX':     '^GDAXI',
}

def to_date_index(series):
    """Remove timezone do index, necessario para alinhar bolsas de fusos diferentes.
    yfinance retorna B3 em America/Sao_Paulo, S&P em America/New_York etc.
    Sem essa normalizacao o pandas nao consegue alinhar e retorna NaN.
    """
    s = series.copy()
    s.index = pd.DatetimeIndex(s.index.date)
    return s

retornos_globais = {'WEGE3': to_date_index(df['wege3'])}
for nome, ticker_idx in indices_globais.items():
    dados_idx = yf.Ticker(ticker_idx).history(period=PERIODO)
    ret_idx = dados_idx['Close'].pct_change().dropna()
    retornos_globais[nome] = to_date_index(ret_idx)

df_global = pd.DataFrame(retornos_globais).dropna()
correlacoes = df_global.corr()['WEGE3'].drop('WEGE3')

def interpretar_correlacao(corr: float) -> str:
    """Retorna 'Alta', 'Moderada' ou 'Baixa' baseado nos benchmarks acima."""
    if abs(corr) > 0.7:
        return 'Alta'
    elif abs(corr) >= 0.3:
        return 'Moderada'
    else:
        return 'Baixa'

print(f'Dias alinhados entre mercados: {len(df_global)}')
print()
print('Correlacao de WEGE3 com indices globais:')
print()
for indice, corr in correlacoes.items():
    nivel = interpretar_correlacao(corr)
    print(f'  {indice:<10} {corr:+.4f}  ({nivel})')

Dias alinhados entre mercados: 221

Correlacao de WEGE3 com indices globais:

  S&P 500    +0.2596  (Baixa)
  Nikkei     +0.0537  (Baixa)
  FTSE       +0.1836  (Baixa)
  DAX        +0.2590  (Baixa)


## Resumo

In [9]:
print('=' * 50)
print(f'  Painel de Risco: {TICKER}')
print('=' * 50)
print(f'  Sharpe Ratio:       {sharpe_anualizado:.2f}  ({interpretar_sharpe(sharpe_anualizado)})')
print(f'  Sortino Ratio:      {sortino_anualizado:.2f}')
print(f'  Max Drawdown:       {mdd:.1%}')
print(f'  VaR 95% (diario):   {var_95:.2%}')
print(f'  Beta:               {beta:.4f}')
print(f'  Alpha (anual):      {alpha:.1%}')
print('=' * 50)

  Painel de Risco: WEGE3.SA
  Sharpe Ratio:       0.27  (Fraco)
  Sortino Ratio:      0.42
  Max Drawdown:       -22.5%
  VaR 95% (diario):   -3.14%
  Beta:               0.2924
  Alpha (anual):      8.8%


## O que virou no projeto

As métricas prototipadas acima viraram a tool `calcular_metricas_risco` (`fluxo_financeiro/tools/financial_data.py`):
Sharpe, Sortino, Max Drawdown, VaR 95% (Monte Carlo com 10.000 simulações e semente fixa, para o número não mudar
a cada execução), Beta e Alpha contra o Ibovespa.

A taxa livre de risco é a **SELIC** do dia, buscada no Banco Central (série 432). Se o BCB não responder em 5 segundos,
a tool usa 10,75% e avisa no log, para a análise não parar por causa disso.

A correlação com índices globais ficou de fora: alguns desses dados vinham bagunçados na API gratuita, e uma métrica montada em cima de dado ruim só confundiria a análise.

In [10]:
from fluxo_financeiro.tools.financial_data import calcular_metricas_risco

print(calcular_metricas_risco.invoke("WEGE3"))

2026-11-06 21:38:04,985 - fluxo_financeiro - INFO - [tool] calcular_metricas_risco: ticker=WEGE3.SA


2026-11-06 21:38:05,504 - fluxo_financeiro - INFO - [tool] calcular_metricas_risco: ok — Sharpe=0.18


Métricas de Risco para WEGE3.SA (último 1 ano):
Referência: Selic 13.75% a.a. (BCB), benchmark IBOV

Sharpe Ratio:      0.18 (Fraco)
Sortino Ratio:     0.28
Max Drawdown:      -22.5%
VaR 95% (diário):  -3.14%
Beta (vs IBOV):    0.29 (defensivo)
Alpha (anual):     6.6% (acima do esperado pelo CAPM)



## O que observar

- A linha "Referência" mostra a SELIC usada: é a base do Sharpe e do Alpha.
- Os rótulos entre parênteses (Bom/OK/Fraco no Sharpe; agressivo/defensivo/neutro no Beta) são as faixas que o agente usa para não interpretar sozinho.